# Amazon Bedrock AgentCore Runtime의 AgentSkills Plugin 기반 Strands Agent

## 개요

이 튜토리얼에서는 필요할 때 전문 지침을 제공하는 `AgentSkills` plugin을 사용해 Strands agent를 빌드하고, 로컬에서 실행한 뒤 Amazon Bedrock AgentCore Runtime에 배포하는 방법을 보여줍니다.

`AgentSkills` plugin은 skill을 정의하는 두 가지 방식, 즉 **file-based**(YAML frontmatter가 포함된 `SKILL.md` 파일)와 **programmatic**(Python에서 inline으로 정의한 `Skill` object)을 지원합니다. 두 방식을 같은 에이전트에서 함께 사용할 수 있습니다.

두 가지 예제 skill을 살펴봅니다. **weather-reporter** skill은 file-based `SKILL.md`로 정의하고, **math-tutor** skill은 `Skill` class를 사용해 programmatic 방식으로 정의합니다. 먼저 로컬에서 실험한 뒤 에이전트를 AgentCore Runtime에 배포합니다.

### 튜토리얼 세부 정보

| 항목 | 값 |
|---|---|
| 튜토리얼 유형 | Jupyter Notebook |
| 에이전트 유형 | AgentSkills Plugin 기반 Strands Agent |
| Agentic Framework | Strands Agents |
| LLM 모델 | Amazon Bedrock(Claude Haiku 4.5) |
| 튜토리얼 구성 요소 | Skill(SKILL.md), AgentSkills Plugin, 로컬 실험, AgentCore Runtime 배포 |
| 튜토리얼 분야 | 일반 / 개발자 교육 |
| 예제 난이도 | 중간 |
| SDK used | bedrock-agentcore-starter-toolkit, strands-agents |

### 튜토리얼 아키텍처

에이전트는 `skills/` 디렉터리를 scan하는 `AgentSkills` plugin을 사용합니다. 각 skill은 YAML frontmatter(`name`, `description`, `allowed-tools`)와 Markdown 지침이 포함된 `SKILL.md` 파일입니다. 에이전트는 사용자 요청에 따라 적절한 skill을 선택합니다.

다음 두 가지 예제 skill을 제공합니다.
- **weather-reporter**: file-based `SKILL.md`로 정의합니다. custom `@tool` weather 함수와 함께 사용하여 날씨 정보를 emoji, 온도 범위, 활동 추천 형식으로 구성합니다.
- **math-tutor**: `Skill` class를 사용하여 programmatic 방식으로 정의합니다(파일 불필요). `strands-agents-tools`의 `calculator` tool과 함께 사용하여 모든 풀이 과정을 보여주며 수학 문제를 단계별로 해결합니다.

### 튜토리얼 주요 기능

* 두 가지 skill 정의 방식: file-based(`SKILL.md`) 및 programmatic(`Skill` class)
* 하나의 `AgentSkills` plugin에서 file-based 및 programmatic skill 함께 사용
* 사용자 의도에 따라 필요한 skill 활성화
* cloud 배포 전 로컬 실험
* Amazon Bedrock AgentCore Runtime에 원활하게 배포


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* 구성된 AWS credentials(`aws configure`)
* Amazon Bedrock AgentCore SDK (`bedrock-agentcore`, `bedrock-agentcore-starter-toolkit`)
* Strands Agents (`strands-agents`, `strands-agents-tools`)
* Amazon Bedrock 모델(Claude Haiku 4.5) 액세스 권한
* Docker 또는 Finch(AgentCore Runtime 배포용)


In [ ]:
%pip install -r requirements.txt

## Skill 생성 방식

`AgentSkills` plugin은 skill을 정의하는 두 가지 방식을 지원합니다. 이 튜토리얼에서는 두 방식을 모두 살펴봅니다.

### 방식 1: File-based skill(SKILL.md)

file-based skill은 YAML frontmatter와 Markdown 지침 본문이 포함된 `SKILL.md` 파일을 담은 디렉터리입니다. 이 방식은 version control하거나 project 간에 공유하거나 Python 코드를 수정하지 않고 편집하려는 skill에 적합합니다.

```
skills/
└── weather-reporter/
    └── SKILL.md
```

**SKILL.md 형식:**
```markdown
---
name: skill-name
description: Short description used by the agent to decide when to activate this skill
allowed-tools:
  - tool_name
---

# Skill Instructions

Markdown body with behavioral instructions for the agent.
```

YAML frontmatter field는 다음과 같습니다.
- `name`: skill의 고유 identifier(kebab-case)
- `description`: 에이전트가 skill 활성화 시점을 결정하는 데 사용하는 사람이 읽을 수 있는 설명
- `allowed-tools`: skill에서 사용하도록 허용된 tool 이름 목록


In [ ]:
import os

os.makedirs("skills/weather-reporter", exist_ok=True)

In [ ]:
%%writefile skills/weather-reporter/SKILL.md
---
name: weather-reporter
description: Format weather information with emoji, temperature ranges, and activity recommendations
allowed-tools:
  - weather
---

# Weather Reporter 지침

You are a friendly weather reporter. When presenting weather information:

1. **Use weather emoji** to make the report visually engaging:
   - ☀️ for sunny/clear conditions
   - 🌧️ for rain
   - ⛅ for partly cloudy
   - 🌩️ for thunderstorms
   - ❄️ for snow
   - 🌫️ for fog/mist

2. **Include temperature ranges** in both Fahrenheit and Celsius

3. **Provide activity recommendations** based on the conditions:
   - Suggest outdoor activities for good weather
   - Recommend indoor alternatives for bad weather
   - Include clothing suggestions (e.g., "bring an umbrella", "wear a light jacket")

4. **Format your response** as a friendly weather report with clear sections for current conditions and recommendations.


### 방식 2: Programmatic skill(Skill class)

programmatic skill은 `Skill` class를 사용하여 Python에서 inline으로 정의하므로 디렉터리나 파일이 필요하지 않습니다. 간단한 skill, 동적 skill 생성 또는 모든 내용을 하나의 script에 유지하려는 경우 편리합니다.

```python
from strands import Skill

math_tutor = Skill(
    name="math-tutor",
    description="Solve math problems step-by-step, showing all work",
    instructions="Break down math problems into clear steps. Show all intermediate calculations and explain your reasoning at each step.",
)
```

두 skill type 모두 동일한 `skills` 목록을 통해 `AgentSkills`에 전달하므로 file-based skill과 programmatic skill을 자유롭게 함께 사용할 수 있습니다.

```python
from strands import AgentSkills

plugin = AgentSkills(skills=["./skills/weather-reporter", math_tutor])
```

아래 agent script에서 `math-tutor` skill을 programmatic 방식으로 정의합니다.


## 로컬 에이전트 실험

AgentCore Runtime에 배포하기 전에 에이전트를 로컬에서 실행하여 skill이 예상대로 작동하는지 확인합니다.

에이전트는 Claude Haiku 4.5 기반 `BedrockModel`을 사용하며 `AgentSkills` plugin을 통해 `skills/` 디렉터리에서 skill을 불러옵니다. 로컬에서 실행하면 전체 배포 cycle 없이 skill 정의와 에이전트 동작을 빠르게 반복할 수 있습니다.


## Amazon Bedrock AgentCore Runtime에 배포

에이전트가 로컬에서 작동하는 것을 확인했으므로 Amazon Bedrock AgentCore Runtime에 배포합니다. 배포용 agent script는 동일한 logic을 `/invocations`와 `/ping` HTTP endpoint를 노출하는 `BedrockAgentCoreApp`으로 감쌉니다.

`bedrock-agentcore-starter-toolkit`은 agent script, `skills/` 디렉터리, 모든 dependency를 Docker container로 package하여 Amazon ECR에 push하고 AgentCore Runtime endpoint를 자동으로 생성합니다.

> **참고:** 배포를 진행하기 전에 Docker 또는 Finch가 실행 중인지 확인합니다.


In [ ]:
import os
import boto3

AWS_PROFILE = "REPLACE-ME"
REGION = "REPLACE-ME"

os.environ["AWS_PROFILE"] = AWS_PROFILE
os.environ["AWS_DEFAULT_REGION"] = REGION

# 기본 session을 reset하여 boto3가 새 profile을 사용하도록 설정
boto3.setup_default_session(profile_name=AWS_PROFILE, region_name=REGION)

# credentials 유효성 확인
identity = boto3.client("sts").get_caller_identity()
print(f"Account: {identity['Account']}, Region: {REGION}")

In [ ]:
%%writefile agent_with_skills.py
from strands import Agent, tool, AgentSkills, Skill
from strands_tools import calculator
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

@tool
def weather() -> dict:
    """Get current weather information."""
    return {
        "condition": "sunny",
        "temp_f": 75,
        "temp_c": 24,
        "humidity": 45,
        "wind_mph": 10
    }

# Programmatic skill - inline으로 정의하며 SKILL.md 파일 불필요
math_tutor = Skill(
    name="math-tutor",
    description="Solve math problems step-by-step, showing all work and explaining reasoning",
    instructions=(
        "You are a patient and thorough math tutor. When solving math problems:\n"
        "1. Break down the problem into clear, numbered steps.\n"
        "2. Show all intermediate calculations — never skip steps.\n"
        "3. Explain your reasoning at each step, identifying the mathematical concept applied.\n"
        "4. Use the calculator tool for arithmetic to ensure accuracy.\n"
        "5. Verify your answer against the original problem.\n"
        "6. Summarize with a clear final answer."
    ),
)

model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")
# file-based skill(weather-reporter)과 programmatic skill(math_tutor) 함께 사용
skills = AgentSkills(skills=["./skills/weather-reporter", math_tutor])
agent = Agent(model=model, tools=[calculator, weather], plugins=[skills])

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke_agent(payload: dict) -> str:
    response = agent(payload.get("prompt", ""))
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()


## AgentCore Runtime 구성 및 실행

`bedrock-agentcore-starter-toolkit`의 `Runtime` class를 사용하여 배포를 구성합니다. `auto_create_execution_role=True`와 `auto_create_ecr=True`를 설정하면 toolkit이 필요한 IAM role과 ECR repository를 자동으로 생성합니다.

`launch()` 호출은 Docker image를 빌드하고 ECR에 push한 뒤 AgentCore Runtime endpoint를 생성합니다. 이 작업에는 몇 분이 걸릴 수 있습니다.


In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="agent_with_skills.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    agent_name="strands_skills_tutorial",
    region=REGION,
)
print("Configuration complete.")

In [ ]:
launch_result = agentcore_runtime.launch()
print("Launch initiated. Waiting for runtime to become READY...")

In [ ]:
import time

terminal_states = ["READY", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"]

status_response = agentcore_runtime.status()
current_status = status_response.endpoint["status"]
print(f"Status: {current_status}")

while current_status not in terminal_states:
    time.sleep(30)
    status_response = agentcore_runtime.status()
    current_status = status_response.endpoint["status"]
    print(f"Status: {current_status}")

if current_status == "READY":
    print("Runtime is READY. Proceeding to invocation.")
else:
    print(f"Deployment ended with status: {current_status}. Check CloudWatch logs for details.")

## Weather Skill 테스트

In [ ]:
# skill 활성화를 유도하는 prompt로 배포된 Runtime 호출
invoke_payload = {
    "prompt": "What's the weather like today? Give me a full report with emoji and activity recommendations."
}
response = agentcore_runtime.invoke(invoke_payload)
response_text = "".join(response["response"])
print(response_text)

In [ ]:
from IPython.display import Markdown, display

# 응답 text 추출 및 표시
response_text = "".join(response["response"])
# 바깥쪽 따옴표를 제거하고 escape된 newline 교체
response_text = response_text.strip('"').replace("\\n", "\n")
display(Markdown(response_text))

## Math Skill 테스트

In [ ]:
# skill 활성화를 유도하는 prompt로 배포된 Runtime 호출
invoke_payload = {"prompt": "help me solve 23+458*89"}
response = agentcore_runtime.invoke(invoke_payload)
response_text = "".join(response["response"])
print(response_text)

## 리소스 정리

이제 AgentCore Runtime과 관련 리소스를 정리합니다. 불필요한 비용을 방지하도록 Runtime을 먼저 삭제한 뒤 ECR repository 같은 지원 리소스를 정리합니다.

In [ ]:
import boto3

agent_name = "strands_skills_tutorial"

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

# 이 셀을 독립적으로 실행하는 경우 이름으로 agent_id 조회
runtimes = agentcore_control_client.list_agent_runtimes()
agent_id = next(
    (r["agentRuntimeId"] for r in runtimes["agentRuntimes"] if r["agentRuntimeName"] == agent_name),
    None,
)

if not agent_id:
    print(f"⚠️ No runtime found with name '{agent_name}'")
else:
    # AgentCore Runtime 삭제
    try:
        agentcore_control_client.delete_agent_runtime(agentRuntimeId=agent_id)
        print(f"✅ Runtime '{agent_name}' ({agent_id}) deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete runtime: {e}")

In [ ]:
ecr_client = boto3.client("ecr", region_name=REGION)
REPO_NAME = "bedrock-agentcore-strands_skills_tutorial"
# REPO_NAME = launch_result.ecr_uri.split('/')[1]

# 배포 중 생성된 ECR repository 삭제
try:
    ecr_client.delete_repository(repositoryName=REPO_NAME, force=True)
    print(f"✅ ECR repository '{REPO_NAME}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")